# Advanced Extension: BERT Transformer Model

First we start with the same data loading as we did with the CNN baseline model. For the advanced extension, we are training a separate BERT model using pre-trained weights and evaluating whether it performed better.

In [26]:
import pandas as pd
from pathlib import Path

data_path = Path("data/clean_data.csv")

df = pd.read_csv(data_path)

df = df.drop_duplicates(subset=['Description', 'Type']) # Drop duplicates between text and label
df = df.drop_duplicates(subset=['Description'], keep=False) # Drop duplicate text with different labels

texts = df["Description"].values # Features
labels = df["Type"].values # Targets

print(df["Type"].value_counts())

Type
Grievance             6482
Required Support      2192
Enquiry               1186
Complaint              547
Suggestion              82
Willing to Support      20
Appreciation            10
Name: count, dtype: int64


Just like with the CNN baseline, we'll encode the target variable, the sentiments, the exact same way with integer values.

In [15]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(labels)

# Print the new encoded labels
print(y)

[2 3 4 ... 1 5 3]


We also want the data splits to be the same so we'll use the same seed that we did to make the splits with the baseline model and do the same split proportions for the train, validation, and test set.

In [27]:
from sklearn.model_selection import train_test_split

# Train/Test split
x_train, x_test, y_train, y_test = train_test_split(
    texts, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

# Train/Validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=42
)

print(f"Number of observations in training set: " + str(len(x_train)))
print(f"Number of observations in validation set: " + str(len(x_val)))
print(f"Number of observations in test set: " + str(len(x_test)))

Number of observations in training set: 7599
Number of observations in validation set: 1342
Number of observations in test set: 1578


The number of observations in each split match exactly to the CNN Baseline split.

With a pre-trained BERT model, the vocabulary comes from its existing vocbaulary, whereas the CNN baseline vocabulary came from the training data.

In [29]:
from transformers import AutoTokenizer

# Loads BERT's fixed vocabulary (30,522 tokens)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

30522


The BERT transformer model has a vocabulary size of 30,522 subword tokens in comparison to the vocabulary size of 14449 word tokens that the CNN baseline model has. Now that the vocabulary is established, we can tokenize all three data splits the same way.

In [30]:
# Create a function to tokenize the text and match it with its respective label
def make_dataset(texts, labels):
    enc = tokenizer(list(texts), truncation=True, max_length=128)
    return [
        {"input_ids": ids, "attention_mask": mask, "labels": int(lab)}
        for ids, mask, lab in zip(enc["input_ids"], enc["attention_mask"], labels)
    ]

# Tokenize each split
train_ds = make_dataset(x_train, y_train)
val_ds   = make_dataset(x_val, y_val)
test_ds  = make_dataset(x_test, y_test)

Now that all the text is tokenized, each batch can be dynamically padded and we can begin the training loop.

In [31]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoConfig, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, set_seed, get_linear_schedule_with_warmup)

def train_bert(train_ds, val_ds, num_labels, from_scratch=False,
               epochs=3, batch_size=16, lr=2e-5):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Dynamic padding per batch
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collator)

    # Pick the model: pretrained or from scratch
    if from_scratch:
        config = AutoConfig.from_pretrained("bert-base-uncased", num_labels=num_labels)
        model = AutoModelForSequenceClassification.from_config(config)
    else:
        model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    model.to(device)

    # Optimize using AdamW
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    best_f1, best_state = -1.0, None # set the best f1 and the best set of weights
    for epoch in range(epochs):
        # Training loop
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        # Validation loop: measuring validation loss and f1
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                preds += model(**batch).logits.argmax(-1).cpu().tolist()
                labels += batch["labels"].cpu().tolist()
        f1 = f1_score(labels, preds, average="macro")
        print(f"Epoch {epoch+1} | val acc {accuracy_score(labels, preds):.4f} | val macro-F1 {f1:.4f}")

        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.clone() for k, v in model.state_dict().items()} # keep best model

    model.load_state_dict(best_state)
    return model

The training loop uses the same loss function, nn.CrossEntropyLoss(), that the baseline CNN does. However, this training loop utilizes AdamW with weight decay for optimization unlike the CNN which used regular Adam.

In [ ]:
# Extract the number of classes
num_labels = len(set(y_train))

# Save the best model
model_pre = train_bert(train_ds, val_ds, num_labels=num_labels)

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | val acc 0.6274 | val macro-F1 0.1641
Epoch 2 | val acc 0.6200 | val macro-F1 0.1821
Epoch 3 | val acc 0.6118 | val macro-F1 0.1859


While the validation accuracy got worse with each epoch, the validation macro-F1 increased. This indicates that the model got better at predicting some of the minority classes since the data is unbalanced, but mislabeled some of the majority class since the majority-class baseline is at 61.6% and the validationa accuracy dipped below that in the last epoch.

Now that the best model has been extracted, we can apply the model to the test set and evaluate how it performs.

In [33]:
from sklearn.metrics import classification_report, confusion_matrix

device = next(model_pre.parameters()).device
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))

model_pre.eval()
preds, labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds += model_pre(**batch).logits.argmax(-1).cpu().tolist()
        labels += batch["labels"].cpu().tolist()

print(f"Test accuracy: {accuracy_score(labels, preds):.4f}")
print(f"Test macro-F1: {f1_score(labels, preds, average='macro', zero_division=0):.4f}\n")
print(classification_report(labels, preds, digits=4, zero_division=0))
print(confusion_matrix(labels, preds))

Test accuracy: 0.6324
Test macro-F1: 0.1918

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.0000    0.0000    0.0000        82
           2     0.5417    0.4382    0.4845       178
           3     0.6606    0.9290    0.7721       972
           4     0.2537    0.0517    0.0859       329
           5     0.0000    0.0000    0.0000        12
           6     0.0000    0.0000    0.0000         3

    accuracy                         0.6324      1578
   macro avg     0.2080    0.2027    0.1918      1578
weighted avg     0.5209    0.6324    0.5482      1578

[[  0   0   2   0   0   0   0]
 [  0   0   4  76   2   0   0]
 [  0   0  78  89  11   0   0]
 [  0   0  32 903  37   0   0]
 [  0   0  26 286  17   0   0]
 [  0   0   2  10   0   0   0]
 [  0   0   0   3   0   0   0]]


# Comparison to CNN Baseline

| Model | Test Macro-F1 | Test Accuracy |
|-------|--------|--------------|
| CNN | 0.1878 | 0.5475 |
| BERT Transformer | 0.1918 | 0.6324 |

| Model | Class | Test F1 |
|-------|-------|----------|
| CNN | Appreciation | 0.0000 |
| BERT Transformer | Appreciation | 0.0000 |
| CNN | Complaint | 0.0714 |
| BERT Transformer | Complaint | 0.0000 |
| CNN | Enquiry | 0.3226 |
| BERT Transformer | Enquiry | 0.4845 |
| CNN | Grievance | 0.7023 |
| BERT Transformer | Grievance | 0.7721 |
| CNN | Required Support | 0.2182 |
| BERT Transformer | Required Support | 0.0859 |
| CNN | Suggestion | 0.0000 |
| BERT Transformer | Suggestion | 0.0000 |
| CNN | Willing to Support | 0.0000 |
| BERT Transformer | Willing to Support | 0.0000 |

Comparison:
* Overall, the BERT Transformer model had a higher test accuracy and higher test macro-f1 score than the baseline CNN model. The CNN model in comparison is the one that scored highest in validation f1 during training.
* The higher macro-f1 score indicates that the BERT Transformer model was better at predicting minority classes. This is also reinforced by the fact that the test accuracy was above the majority-class baseline.
* The BERT Transformer model is better at predicting the Enquiry class and the majority class, Grievance. The CNN is better at predicting the Required Support and the Complaint class. Both models could not correctly predict Appreciation, Suggestion, or Willing to Support.

# Controlled Comparison: BERT Model (from scratch)

For the controlled comparison, the pre-trained BERT model is compared against a BERT model trained from scratch. They both use the same training loops and the only difference is one model has a frozen set of weights whereas the scratch model starts from a random set of initialized weights.

In [ ]:
# Start the training loop for the random weight model
model_scr = train_bert(train_ds, val_ds, num_labels=num_labels, from_scratch=True) # make sure 'from_scratch' is set to True

Epoch 1 | val acc 0.6162 | val macro-F1 0.1089
Epoch 2 | val acc 0.6162 | val macro-F1 0.1089
Epoch 3 | val acc 0.6162 | val macro-F1 0.1089


The BERT model from scratch did not improve with each epoch and the validation accuracy stayed right around the majority class baseline of 61.6% and macro-F1 of 0.1809. Now that the weights are set, we can apply the model to the test set to see if the same result occurs with the test set.

In [25]:
device = next(model_scr.parameters()).device
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))

model_scr.eval()
preds, labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds += model_scr(**batch).logits.argmax(-1).cpu().tolist()
        labels += batch["labels"].cpu().tolist()

print(f"Test accuracy: {accuracy_score(labels, preds):.4f}")
print(f"Test macro-F1: {f1_score(labels, preds, average='macro', zero_division=0):.4f}\n")
print(classification_report(labels, preds, digits=4, zero_division=0))
print(confusion_matrix(labels, preds))

Test accuracy: 0.6160
Test macro-F1: 0.1089

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.0000    0.0000    0.0000        82
           2     0.0000    0.0000    0.0000       178
           3     0.6160    1.0000    0.7624       972
           4     0.0000    0.0000    0.0000       329
           5     0.0000    0.0000    0.0000        12
           6     0.0000    0.0000    0.0000         3

    accuracy                         0.6160      1578
   macro avg     0.0880    0.1429    0.1089      1578
weighted avg     0.3794    0.6160    0.4696      1578

[[  0   0   0   2   0   0   0]
 [  0   0   0  82   0   0   0]
 [  0   0   0 178   0   0   0]
 [  0   0   0 972   0   0   0]
 [  0   0   0 329   0   0   0]
 [  0   0   0  12   0   0   0]
 [  0   0   0   3   0   0   0]]


| Model | Test Macro-F1 | Test Accuracy |
|-------|--------|--------------|
| Majority-Class Baseline | 0.1089 | 0.6160 |
| BERT from scratch | 0.1089 | 0.6160 |
| BERT Transformer | 0.1918 | 0.6324 |

The BERT model trained from random initialized weights could only correctly predict the majority class, Greivance. The test macro-f1 and accuracy align exactly with the majority-class baseline. The BERT Transformer model outperfoms both, indicating that it can successfully predict some of minority classes.

# Conclusion

The BERT Transformer model performed better than the CNN baseline in terms of macro-f1 and accuracy. The gap in macro-F1 wasn't huge with the CNN baseline scoring 0.1878 and the BERT Transformer model scoring 0.1918. When looking closely into the per-class breakdown, the BERT Transformer gained mostly within the Enquiry class with a per-class F1 score of 0.4845 compared to the CNN model's score of 0.3226.

To examine what could have contributed to the BERT Transformer performing better, a controlled comparison was done with a BERT model trained from scratch. This means the BERT Transformer had pre-trained weights (frozen body) whereas the BERT model from scratch was starting out with randomly initialized weights. Since the BERT Transformer out-performed the BERT model from scratch even though everything else about the implementations were the same, this means the pre-trained weights contributed to the BERT Transformer's success. The pretraining meant the BERT Transformer model already understood word meanings and grammar. While the CNN model scored better than the BERT from scratch model in macro-F1, the BERT Tranformer beat it out, suggesting again that the improvement came from pretraining.